In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)

print("Groq key loaded:", bool(os.getenv("GROQ_API_KEY")))

Groq key loaded: True


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000206435AAB90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000206437072D0>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
response = llm.invoke(
    "What is revenue growth? Explain in one simple sentence."
)

print(response.content)

Revenue growth is the increase in a company’s total sales or income over a given period.


In [4]:
response

AIMessage(content='Revenue growth is the increase in a company’s total sales or income over a given period.', additional_kwargs={'reasoning_content': 'The user asks: "What is revenue growth? Explain in one simple sentence." So answer: "Revenue growth is the increase in a company\'s sales over a period." Provide a simple sentence.'}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 82, 'total_tokens': 148, 'completion_time': 0.067457716, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.003972005, 'prompt_tokens_details': None, 'queue_time': 0.211836492, 'total_time': 0.071429721}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e91f-e2d2-7372-82c7-e026a21a3a02-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 82, 'output_tokens': 66, 'total_tokens': 148, 'output_token_detai

In [10]:
from langchain_core.prompts import PromptTemplate

prompt = """
You are a financial information extraction assistant.

From the text below, extract the following fields:

- company
- revenue_actual
- revenue_expected
- eps_actual
- eps_expected

Rules:
1. Return only valid JSON.
2. Do not add any explanation or markdown.
3. If a value is missing, use "Not Mentioned".
4. Keep the unit with the value when available.

Text:
{text}
"""


pt = PromptTemplate.from_template(prompt)

In [6]:
article = """
ABC Ltd reported quarterly revenue of $1.25 billion,
compared with analyst expectations of $1.18 billion.
EPS came in at $2.40 versus expectations of $2.15.
"""

formatted_prompt = pt.invoke({"text": article})

formatted_prompt

StringPromptValue(text='\nYou are a financial information extraction assistant.\n\nFrom the text below, extract:\n- company\n- revenue_actual\n- revenue_expected\n- eps_actual\n- eps_expected\n\nIf a value is not mentioned, return "Not Mentioned".\n\nText:\n\nABC Ltd reported quarterly revenue of $1.25 billion,\ncompared with analyst expectations of $1.18 billion.\nEPS came in at $2.40 versus expectations of $2.15.\n\n')

In [7]:
print(formatted_prompt.text)


You are a financial information extraction assistant.

From the text below, extract:
- company
- revenue_actual
- revenue_expected
- eps_actual
- eps_expected

If a value is not mentioned, return "Not Mentioned".

Text:

ABC Ltd reported quarterly revenue of $1.25 billion,
compared with analyst expectations of $1.18 billion.
EPS came in at $2.40 versus expectations of $2.15.




In [11]:
chain = pt | llm

response = chain.invoke({"text": article})

print(response.content)

{"company":"ABC Ltd","revenue_actual":"$1.25 billion","revenue_expected":"$1.18 billion","eps_actual":"$2.40","eps_expected":"$2.15"}


In [13]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

In [14]:
result = parser.parse(response.content)

result

{'company': 'ABC Ltd',
 'revenue_actual': '$1.25 billion',
 'revenue_expected': '$1.18 billion',
 'eps_actual': '$2.40',
 'eps_expected': '$2.15'}

In [15]:
print(type(response.content))
print(type(result))

<class 'str'>
<class 'dict'>


In [16]:
result["company"]

'ABC Ltd'

In [17]:
chain = pt | llm | parser

In [18]:
result = chain.invoke({"text": article})

result

{'company': 'ABC Ltd',
 'revenue_actual': '$1.25 billion',
 'revenue_expected': '$1.18 billion',
 'eps_actual': '$2.40',
 'eps_expected': '$2.15'}

In [19]:
type(result)

dict

In [20]:
article_2 = """
XYZ Ltd reported revenue of ₹850 crore,
up 22% year over year. The company also announced
a new manufacturing facility with an investment of ₹200 crore.
"""

result_2 = chain.invoke({"text": article_2})

result_2

{'company': 'XYZ Ltd',
 'revenue_actual': '₹850 crore',
 'revenue_expected': 'Not Mentioned',
 'eps_actual': 'Not Mentioned',
 'eps_expected': 'Not Mentioned'}

In [21]:
prompt = """
You are a financial information extraction assistant.

From the financial text below, extract the following information:

1. company
2. reporting_period
3. revenue_actual
4. revenue_expected
5. revenue_growth
6. eps_actual
7. eps_expected
8. profit_growth
9. margin
10. margin_change
11. capex
12. capacity_expansion
13. new_product_or_business
14. major_order_or_deal
15. management_guidance
16. key_positive
17. key_risk

Rules:
- Extract only information explicitly mentioned in the text.
- Do not guess or invent information.
- If a field is not mentioned, return "Not Mentioned".
- Keep units such as crore, million, billion, %, etc.
- Keep YoY or QoQ labels where mentioned.
- Return only valid JSON.
- Do not add markdown or explanation.

Financial Text:
{text}
"""

pt = PromptTemplate.from_template(prompt)

chain = pt | llm | parser

In [22]:
article_3 = """
ABC Electronics Ltd reported Q2 FY27 revenue of ₹1,250 crore,
up 18% year-on-year, compared with analyst expectations of ₹1,180 crore.

EPS came in at ₹12.40 versus expectations of ₹11.80.

EBITDA margin improved to 16.2% from 14.8% in the same quarter last year.

The company announced a ₹300 crore investment to expand its manufacturing
capacity by 25% and plans to launch two new EV-related products next quarter.

Management expects FY27 revenue growth of 15% to 18%.

However, management highlighted higher raw material prices as a near-term risk.
"""

result_3 = chain.invoke({"text": article_3})

result_3

{'company': 'ABC Electronics Ltd',
 'reporting_period': 'Q2 FY27',
 'revenue_actual': '₹1,250 crore',
 'revenue_expected': '₹1,180 crore',
 'revenue_growth': 'up 18% year-on-year',
 'eps_actual': '₹12.40',
 'eps_expected': '₹11.80',
 'profit_growth': 'Not Mentioned',
 'margin': '16.2%',
 'margin_change': 'improved to 16.2% from 14.8%',
 'capex': '₹300 crore',
 'capacity_expansion': '25% increase',
 'new_product_or_business': 'launch of two new EV-related products next quarter',
 'major_order_or_deal': 'Not Mentioned',
 'management_guidance': 'expects FY27 revenue growth of 15% to 18%',
 'key_positive': 'Revenue up 18% YoY, EBITDA margin improved to 16.2%, capacity expansion by 25%, launch of two new EV products',
 'key_risk': 'higher raw material prices as a near-term risk'}

In [23]:
from langchain_core.exceptions import OutputParserException

def extract_financial_data(text):
    try:
        result = chain.invoke({"text": text})
        return result

    except OutputParserException as e:
        return {
            "error": "The model returned an invalid structured response.",
            "details": str(e)
        }

    except Exception as e:
        return {
            "error": "Something went wrong while processing the text.",
            "details": str(e)
        }

In [24]:
extracted_data = extract_financial_data(article_3)

extracted_data

{'company': 'ABC Electronics Ltd',
 'reporting_period': 'Q2 FY27',
 'revenue_actual': '₹1,250 crore',
 'revenue_expected': '₹1,180 crore',
 'revenue_growth': 'up 18% year-on-year',
 'eps_actual': '₹12.40',
 'eps_expected': '₹11.80',
 'profit_growth': 'Not Mentioned',
 'margin': '16.2%',
 'margin_change': 'improved to 16.2% from 14.8%',
 'capex': '₹300 crore',
 'capacity_expansion': '25% increase in manufacturing capacity',
 'new_product_or_business': 'launch of two new EV-related products next quarter',
 'major_order_or_deal': 'Not Mentioned',
 'management_guidance': 'expects FY27 revenue growth of 15% to 18%',
 'key_positive': 'Revenue up 18% YoY, EBITDA margin improved to 16.2%, capacity expansion by 25%, launch of two new EV products',
 'key_risk': 'higher raw material prices as a near-term risk'}

In [25]:
import pandas as pd

df = pd.DataFrame(
    extracted_data.items(),
    columns=["Field", "Value"]
)

df

,Field,Value
0,company,ABC Electronics Ltd
1,reporting_period,Q2 FY27
2,revenue_actual,"₹1,250 crore"
3,revenue_expected,"₹1,180 crore"
4,revenue_growth,up 18% year-on-year
5,eps_actual,₹12.40
6,eps_expected,₹11.80
7,profit_growth,Not Mentioned
8,margin,16.2%
9,margin_change,improved to 16.2% from 14.8%


In [26]:
df["Field"] = (
    df["Field"]
    .str.replace("_", " ")
    .str.title()
)

df

,Field,Value
0,Company,ABC Electronics Ltd
1,Reporting Period,Q2 FY27
2,Revenue Actual,"₹1,250 crore"
3,Revenue Expected,"₹1,180 crore"
4,Revenue Growth,up 18% year-on-year
5,Eps Actual,₹12.40
6,Eps Expected,₹11.80
7,Profit Growth,Not Mentioned
8,Margin,16.2%
9,Margin Change,improved to 16.2% from 14.8%


In [27]:
def extract_to_dataframe(text):
    data = extract_financial_data(text)

    if "error" in data:
        return data

    df = pd.DataFrame(
        data.items(),
        columns=["Field", "Value"]
    )

    df["Field"] = (
        df["Field"]
        .str.replace("_", " ")
        .str.title()
    )

    return df

In [28]:
extract_to_dataframe(article_3)

,Field,Value
0,Company,ABC Electronics Ltd
1,Reporting Period,Q2 FY27
2,Revenue Actual,"₹1,250 crore"
3,Revenue Expected,"₹1,180 crore"
4,Revenue Growth,up 18% year-on-year
5,Eps Actual,₹12.40
6,Eps Expected,₹11.80
7,Profit Growth,Not Mentioned
8,Margin,16.2%
9,Margin Change,improved to 16.2% from 14.8%
